# B. Sequence → MMseqs2 search → clusters → alignment → tree

Paste a protein sequence into the **Parameters** cell, point `DATABASE` at a database made by **notebook A**, and run all cells (Runtime → Run all). The code cells are collapsed (they are Colab forms); the text above each one says what it does. Click a cell's title and "Show code" if you want to read or change it.

Building the databases (the slow part: downloading and converting UniRef, selecting taxa, packaging) is done once in notebook A. This notebook only loads a finished database folder (or a `.tar` of one).

## What the notebook does, step by step
| Step | What happens | Typical cost | Output you will see |
|---|---|---|---|
| 1 Environment | Installs MMseqs2, MAFFT, tree program, Biopython | A minute or two (not measured) | Tool versions, free disk |
| 2 Query | Checks and writes your sequence | Instant | Query length |
| 3 Database | Unpacks and checks the database, copies it to local disk if it is on Drive | Seconds for a folder on local disk; copying from Drive depends on size (not measured) | Name, sequence and taxid counts, origin |
| 4 Search | MMseqs2 search of the database with your query | Not measured on real data; grows with database size and number of iterations | Progress lines, hit file size |
| 5 Filter | Drops duplicates, hits without taxonomy and short alignments, then samples | Seconds | Filter log table, group counts |
| 6 Cluster | Clusters the hits at several identity thresholds and checks them against species | Seconds to minutes for a few thousand sequences | Threshold table, three plots, cluster summary |
| 7 Reduce | Removes near-duplicates so the tree stays small | Seconds to a minute | Size of the tree set |
| 8 Align | MAFFT multiple alignment | Minutes for hundreds of sequences (not measured) | Occupancy plot |
| 9 Tree | FastTree, IQ-TREE or neighbour joining | FastTree about a minute for hundreds of sequences; IQ-TREE is much slower (not measured) | Rooting line |
| 10-11 Plot, summary | Draws the tree, lists nearest tips | Seconds | Figure, tables, a short text |
| 12 Stage 2 (optional) | Sorts the sequences of a second, complete-proteome database into the stage 1 classes | Not measured | Species-by-class table and plots |

## Search time
A search reads the whole database on every iteration, so the database you pick decides the cost. `SEARCH_PRESET = "quick"` uses one iteration and a lower sensitivity; move to `standard` or `sensitive` after a first look.

## Checkpoints
- Every step saves its output with the parameters that produced it in `CKPT_DIR/<step>/` and offers a zip to download (`DOWNLOAD_CKPT`).
- Re-running a cell loads its checkpoint if the parameters match. If they differ, the cell lists which ones, and recomputes (cheap steps) or stops (the search). Tick `OVERRIDE_CHECKPOINT` to load the old checkpoint anyway, or `FORCE_RERUN` to recompute and overwrite it.
- A search checkpoint is tied to the database's identity (from `db_info.json`), not to its path, so it still matches after you move the database.
- After a lost session, tick `UPLOAD_CKPT` and upload the zips you downloaded.

## Things to keep in mind
- Use a CPU runtime. High-RAM helps with large databases. A GPU is not used.
- In UniRef databases each entry represents a cluster and its taxonomy is that of the representative member. Two UniRef50 entries from the same species are less than 50% identical by construction, which limits what the clustering check can show at higher identity thresholds.
- Remote homologs of other families (paralogs) can match your query. The clustering step is there to show them.
- A tree built from a database search reflects database sampling, not the true diversity of life.

In [ ]:
#@title Parameters { display-mode: "form" }
#@markdown ### Query
QUERY_NAME = "my_protein" #@param {type:"string"}
QUERY_SEQ = "" #@param {type:"string"}
#@markdown ### Database (made in notebook A)
#@markdown `DATABASE` is a database folder (it contains `db_info.json`) or a `.tar` package of one. If it is under Google Drive and `LOCAL_COPY` is ticked, it is copied to local disk first, because searching straight from Drive is slow.
DATABASE = "" #@param {type:"string"}
LOCAL_COPY = True #@param {type:"boolean"}
LOCAL_DIR = "/content/mmseqs_local" #@param {type:"string"}
WORKDIR = "/content/work" #@param {type:"string"}
#@markdown ### Search
#@markdown `quick`: sensitivity 5.7, one iteration, up to 2000 hits. `standard`: 7.0, one iteration, 10000 hits. `sensitive`: 7.5, two iterations, 20000 hits. `manual` uses the three values below.
SEARCH_PRESET = "quick" #@param ["quick", "standard", "sensitive", "manual"]
EVALUE = 1e-5 #@param {type:"number"}
MANUAL_SENSITIVITY = 7.0 #@param {type:"number"}
MANUAL_ITERATIONS = 1 #@param {type:"integer"}
MANUAL_MAX_SEQS = 10000 #@param {type:"integer"}
SPLIT_MEMORY_LIMIT = "8G" #@param {type:"string"}
#@markdown ### Filtering and sampling of the hits
MIN_QCOV = 0.5 #@param {type:"number"}
MIN_FIDENT = 0.0 #@param {type:"number"}
TAXON_SCOPE = "" #@param {type:"string"}
#@markdown `SUBSET_N` limits everything downstream to roughly this many sequences (0 = no limit). By species, whole species are drawn at random until at least `SUBSET_N` sequences are reached (so the result can be larger); by sequence, exactly `SUBSET_N` are drawn. Species sampling keeps the sequences of a species together, which the clustering check needs.
SUBSET_N = 1000 #@param {type:"integer"}
SUBSET_BY = "species" #@param ["species", "sequence"]
SEED = 1 #@param {type:"integer"}
#@markdown ### Clustering (cheap first look before the tree)
#@markdown Identity thresholds, comma-separated. Anchors are optional labelled reference sequences as `UniProtAccession:label`, fetched from UniProt (leave empty to skip).
CLUSTER_IDS = "0.6,0.5,0.4,0.3,0.25" #@param {type:"string"}
CLUSTER_COV = 0.6 #@param {type:"number"}
CLUSTER_PICK = "auto" #@param {type:"string"}
MIN_SPECIES_PER_CLUSTER = 5 #@param {type:"integer"}
ANCHORS = "" #@param {type:"string"}
TREE_CLUSTER_SCOPE = "all clusters" #@param ["all clusters", "query cluster only"]
#@markdown ### Redundancy reduction and tree size
DEDUP_ID = 0.9 #@param {type:"number"}
MAX_TREE_SEQS = 800 #@param {type:"integer"}
#@markdown ### Tree and plot
TREE_METHOD = "fasttree" #@param ["fasttree", "iqtree", "nj"]
LAYOUT = "circular" #@param ["circular", "rectangular"]
GROUP_RANK = "phylum" #@param ["superkingdom", "kingdom", "phylum", "class", "order", "family", "genus"]
MAX_GROUPS = 8 #@param {type:"integer"}
#@markdown ### Stage 2 (optional): sort the sequences of a second database into the stage 1 classes
#@markdown Stage 2 searches a second database, normally one of complete proteomes made in notebook A, with the sequences of the stage 1 tree set, and gives every hit the class (cluster) of its best-matching query.
STAGE2 = False #@param {type:"boolean"}
STAGE2_DATABASE = "" #@param {type:"string"}
STAGE2_CLASSES = 10 #@param {type:"integer"}
#@markdown ### Checkpoints
CKPT_DIR = "/content/work/checkpoints" #@param {type:"string"}
DOWNLOAD_CKPT = True #@param {type:"boolean"}
UPLOAD_CKPT = False #@param {type:"boolean"}
OVERRIDE_CHECKPOINT = False #@param {type:"boolean"}
FORCE_RERUN = False #@param {type:"boolean"}
KEEP_HISTORY = False #@param {type:"boolean"}

In [ ]:
# Advanced (optional). Leave empty for a generic run.
# TAXON_SCOPE above is a name that must appear in the lineage, e.g. "Fungi" or "Eukaryota".

# Clades that should be shown as their own group when present in a lineage (first match wins),
# even if they are not at GROUP_RANK. Example: ["Pezizomycotina", "Saccharomycotina", "Metazoa"]
EXTRA_CLADES = []

# Optional paralog classification. Each rule is a regular expression searched in the FULL target sequence;
# the first matching rule gives the class, otherwise "unclassified".
# Example: {"E2-like": r"DHR", "E3BP-like": r"..."}   (use rules that suit your family)
PARALOG_RULES = {}

# Name of a class (from PARALOG_RULES), an anchor label (from ANCHORS) or a cluster label such as "C3" to use as outgroup
# for rooting ("" = midpoint rooting)
OUTGROUP_CLASS = ""

# Optional motifs, shown as one ring each (present / absent). Regular expressions on the full target sequence.
# Example: {"my_motif": r"D[IL]{2}D"}
MOTIFS = {}

## 1. Environment
**What happens.** Installs what is missing: the MMseqs2 binary (from its GitHub releases), MAFFT, FastTree or IQ-TREE (from the Linux packages) and Biopython. It also defines the helper that runs shell commands and shows MMseqs2's progress bar about every 20 seconds.

**Expect.** A minute or two, mostly package installation (not measured). Output is the tool versions, the number of CPU threads and free disk space. A free Colab runtime has 2 threads, which is why the large steps are slow.

**Parameters that matter.** `TREE_METHOD` decides which tree program is installed.

In [ ]:
#@title 1. Environment { display-mode: "form" }
import os, sys, re, json, math, shutil, hashlib, subprocess, textwrap, time, platform, gzip, datetime
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
THREADS = os.cpu_count() or 2
WORK = Path(WORKDIR); WORK.mkdir(parents=True, exist_ok=True)
TMP = WORK / "tmp"; TMP.mkdir(exist_ok=True)
BIN = (Path("/content") if IN_COLAB else WORK) / "bin"

PROGRESS_RE = re.compile(r"^\s*\[[=> ]*\]")

def sh(cmd, quiet=False, tail=25, every=20):
    """Run a shell command and stream its output. MMseqs2 progress-bar lines are shown at most once per `every` seconds.
    Raises with the last lines if the command fails."""
    print("$", cmd)
    t0, last = time.time(), 0.0
    p = subprocess.Popen(["bash", "-c", "set -o pipefail; " + cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if quiet:
            continue
        if PROGRESS_RE.match(line):
            if time.time() - last >= every:
                print(f"  [{time.time() - t0:6.0f}s] {line.strip()[:110]}")
                last = time.time()
        else:
            print(line, end="")
    rc = p.wait()
    if rc:
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}\n" + "".join(lines[-tail:]))
    if not quiet:
        print(f"  (finished in {time.time() - t0:.0f}s)")
    return "".join(lines)

def have(x):
    return shutil.which(x) is not None

# To keep the database and checkpoints between sessions, mount Google Drive first and point CKPT_DIR (and DATABASE, if you want to read it from there) at a folder there:
# from google.colab import drive; drive.mount("/content/drive")

# MMseqs2 static binary from the official GitHub releases
if not have("mmseqs"):
    flags = open("/proc/cpuinfo").read()
    variant = "avx2" if " avx2" in flags else ("sse41" if " sse4_1" in flags else "sse2")
    url = f"https://github.com/soedinglab/MMseqs2/releases/latest/download/mmseqs-linux-{variant}.tar.gz"
    sh(f"mkdir -p {BIN} && cd {BIN} && wget -q {url} -O mmseqs.tgz && tar xzf mmseqs.tgz && rm mmseqs.tgz")
    os.environ["PATH"] = f"{BIN}/mmseqs/bin:" + os.environ["PATH"]

# MAFFT and tree programs from the distribution packages
def find_fasttree():
    return shutil.which("FastTree") or shutil.which("fasttree")

def find_iqtree():
    return shutil.which("iqtree2") or shutil.which("iqtree")

missing = []
if not have("mafft"): missing.append("mafft")
if TREE_METHOD == "fasttree" and not find_fasttree(): missing.append("fasttree")
if TREE_METHOD == "iqtree" and not find_iqtree(): missing.append("iqtree")
if missing:
    sh("apt-get update -qq && apt-get install -y -qq " + " ".join(missing), quiet=True)

try:
    import Bio
except ImportError:
    sh("pip install -q biopython", quiet=True)
    import Bio

import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

print("python", platform.python_version(), "| biopython", Bio.__version__, "| pandas", pd.__version__,
      "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print("mmseqs", sh("mmseqs version", quiet=True).strip())
print("mafft", (sh("mafft --version 2>&1 || true", quiet=True).strip().splitlines() or ["?"])[-1])
if TREE_METHOD == "fasttree":
    print("fasttree", find_fasttree())
if TREE_METHOD == "iqtree":
    print("iqtree", find_iqtree())
print("threads:", THREADS, "| free disk (GB) in work dir:", round(shutil.disk_usage(WORK).free / 1e9, 1))

## 1b. Checkpoint helpers
**What happens.** Defines how steps save and reload their results. Each step writes `CKPT_DIR/<step>/meta.json` (its parameters, a hash of them, a timestamp) plus data files. A step's parameters include the key of the checkpoint it was built from, so changing an upstream setting invalidates everything after it. `CKPT_DIR/manifest.json` lists all steps. If `UPLOAD_CKPT` is ticked, this cell also asks you to upload zips saved earlier.

**Expect.** Instant. The output lists the steps already saved in `CKPT_DIR`.

**Parameters that matter.**
- `OVERRIDE_CHECKPOINT`: load a checkpoint even when the parameters differ (the output then belongs to the old parameters, and the cell says so).
- `FORCE_RERUN`: ignore checkpoints and recompute.
- `KEEP_HISTORY`: move old checkpoints into `CKPT_DIR/history` instead of deleting them.
- `DOWNLOAD_CKPT`: download a zip of each step as it finishes. Colab may ask once to allow multiple downloads.

In [ ]:
#@title 1b. Checkpoint helpers { display-mode: "form" }
NB_VERSION = "3"
CK = Path(CKPT_DIR); CK.mkdir(parents=True, exist_ok=True)
STEP_NO = {"search": 2, "filter": 3, "cluster": 4, "reduce": 5, "align": 6, "tree": 7}
KEYS = {}                     # step -> key of the checkpoint actually in use

def keyof(*objs):
    return hashlib.sha1(json.dumps(objs, sort_keys=True, default=str).encode()).hexdigest()[:12]

def _short(v, n=70):
    s = str(v)
    return s if len(s) <= n else s[:n] + f"... ({len(s)} characters)"

def ck_load(step, params, expensive=False):
    """Return the checkpoint's meta dict if a usable one exists, else None (the caller recomputes)."""
    mf = CK / step / "meta.json"
    if FORCE_RERUN:
        print(f"[{step}] FORCE_RERUN is ticked: recomputing and overwriting.")
        return None
    if not mf.exists():
        return None
    meta = json.loads(mf.read_text())
    if meta["key"] == keyof(params):
        print(f"[{step}] using checkpoint from {meta['created']} (parameters match)")
        KEYS[step] = meta["key"]
        return meta
    old = meta["params"]
    diff = [f"    {k}: checkpoint={_short(old.get(k))} | now={_short(params.get(k))}"
            for k in sorted(set(old) | set(params)) if old.get(k) != params.get(k)]
    head = f"[{step}] a checkpoint from {meta['created']} exists but was made with different parameters:\n" + "\n".join(diff)
    if OVERRIDE_CHECKPOINT:
        print(head + "\n    OVERRIDE_CHECKPOINT is ticked: loading the old checkpoint anyway. Results below correspond to the OLD parameters.")
        KEYS[step] = meta["key"]
        return meta
    if expensive:
        raise RuntimeError(head + "\n    This step is slow, so it stops instead of recomputing. Tick OVERRIDE_CHECKPOINT to load the old "
                           "checkpoint anyway, or FORCE_RERUN to recompute and overwrite it.")
    print(head + "\n    Recomputing; the old checkpoint will be " + ("moved to history." if KEEP_HISTORY else "overwritten."))
    return None

def ck_table(step, name):
    return pd.read_csv(CK / step / f"{name}.tsv.gz", sep="\t", dtype=str, keep_default_na=False)

def ck_save(step, params, tables=None, files=None, extra=None):
    d = CK / step
    if d.exists():
        if KEEP_HISTORY:
            (CK / "history").mkdir(exist_ok=True)
            shutil.move(str(d), str(CK / "history" / f"{step}_{datetime.datetime.now():%Y%m%d_%H%M%S}"))
        else:
            shutil.rmtree(d)
    d.mkdir(parents=True)
    for n, df in (tables or {}).items():
        df.to_csv(d / f"{n}.tsv.gz", sep="\t", index=False)
    for n, p in (files or {}).items():
        shutil.copy(p, d / n)
    meta = {"step": step, "notebook_version": NB_VERSION, "created": datetime.datetime.now().isoformat(timespec="seconds"),
            "key": keyof(params), "params": params, "extra": extra or {}}
    (d / "meta.json").write_text(json.dumps(meta, indent=1, default=str))
    KEYS[step] = meta["key"]
    man = [json.loads(m.read_text()) for m in sorted(CK.glob("*/meta.json"))]
    (CK / "manifest.json").write_text(json.dumps([{k: m[k] for k in ("step", "created", "key", "notebook_version")} for m in man], indent=1))
    print(f"[{step}] checkpoint saved: {d}")
    if DOWNLOAD_CKPT:
        name = f"ckpt_{STEP_NO.get(step, 0)}_{step}"
        z = shutil.make_archive(str(WORK / name), "zip", str(CK), step)
        print(f"[{step}] zip: {z} ({os.path.getsize(z) / 1e6:.1f} MB)")
        if IN_COLAB:
            from google.colab import files as colab_files
            colab_files.download(z)

if UPLOAD_CKPT:
    if not IN_COLAB:
        print("UPLOAD_CKPT only works in Colab; copy the unzipped step folders into", CK, "instead.")
    else:
        from google.colab import files as colab_files
        import zipfile
        for fname in colab_files.upload():
            if fname.endswith(".zip"):
                with zipfile.ZipFile(fname) as z:
                    z.extractall(CK)
                print("restored", fname)
print("Checkpoint folder:", CK, "| existing steps:", sorted(p.parent.name for p in CK.glob("*/meta.json")) or "none")

## 2. Query
**What happens.** Cleans your sequence (removes a FASTA header, whitespace and a trailing `*`), checks that it contains only amino-acid letters and has a plausible length (20 to 10000), and writes it to `query.fasta`.

**Expect.** Instant. Output: the cleaned length. An error here means the pasted text is not a protein sequence.

**Parameters that matter.** `QUERY_SEQ`. A short domain and a full-length protein behave very differently later: with `MIN_QCOV` at 0.5, a hit must align over half of *your query*, so a short query finds many partial matches, and a long multi-domain query finds fewer.

In [ ]:
#@title 2. Query { display-mode: "form" }
raw = QUERY_SEQ
seq = "".join(l for l in raw.strip().splitlines() if not l.startswith(">"))
seq = re.sub(r"\s+", "", seq).upper().rstrip("*")
if not seq:
    raise ValueError("Paste a protein sequence into QUERY_SEQ in the Parameters cell.")
bad = sorted(set(seq) - set("ACDEFGHIKLMNPQRSTVWYX"))
if bad:
    raise ValueError(f"QUERY_SEQ contains characters that are not amino-acid letters: {bad}")
if not (20 <= len(seq) <= 10000):
    raise ValueError(f"QUERY_SEQ has length {len(seq)}; expected between 20 and 10000 residues.")
QNAME = re.sub(r"\W+", "_", QUERY_NAME).strip("_") or "query"
QFASTA = WORK / "query.fasta"
QFASTA.write_text(f">{QNAME}\n{seq}\n")
print(f"Query '{QNAME}': {len(seq)} residues, written to {QFASTA}")

## 3. Load the database
**What happens.** Reads `db_info.json` in the database folder (or unpacks a `.tar` package into `LOCAL_DIR` first), checks that all database files are present and that none is a link, copies a database that lives on Google Drive to local disk if `LOCAL_COPY` is ticked, and prints what the database is: name, kind, source, number of sequences and distinct taxids, and the self-test result recorded when it was built.

**Expect.** Seconds for a folder on local disk. Unpacking or copying from Drive takes as long as the data size at your transfer speed (not measured); a few GB is usually minutes. The search later reads the database many times, which is why a local copy matters.

**Parameters that matter.**
- `DATABASE`: which database to search. The choice decides which taxa the search can find, and how long it takes.
- `LOCAL_COPY` and `LOCAL_DIR`: copy to local disk (default on). Untick only if the database is already local or disk is short.

In [ ]:
#@title 3. Load the database { display-mode: "form" }
import tarfile

def n_seqs(db):
    return sum(1 for _ in open(str(db) + ".index"))

def load_database(spec, label="database"):
    """Return (db prefix path, info dict) for a folder made by notebook A, or a .tar package of one."""
    if not spec:
        raise ValueError(f"{label} is empty. Set it to a database folder (or .tar) made by notebook A.")
    p = Path(spec)
    if p.is_file() and p.suffix == ".tar":
        with tarfile.open(p) as tf:
            top = tf.getnames()[0].split("/")[0]
            dest = Path(LOCAL_DIR) / top
            if (dest / "db_info.json").exists():
                print(f"{p.name}: already unpacked in {dest}")
            else:
                Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)
                tf.extractall(LOCAL_DIR)
                print(f"{p.name}: unpacked to {dest}")
        p = dest
    if not (p / "db_info.json").exists():
        raise FileNotFoundError(f"No db_info.json in {p}. Use a folder made by notebook A.")
    on_drive = str(p.resolve()).startswith("/content/drive")
    if on_drive and LOCAL_COPY:
        dest = Path(LOCAL_DIR) / p.name
        if (dest / "db_info.json").exists():
            print(f"Using the local copy {dest}")
        else:
            Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Copying {p} to {dest} (searching straight from Drive is slow)")
            shutil.copytree(p, dest)
        p = dest
    info = json.loads((p / "db_info.json").read_text())
    db = p / info.get("db_prefix", "db")
    for ext in (".dbtype", ".index", ".lookup", "_h", "_h.index", "_mapping", "_taxonomy"):
        if not Path(str(db) + ext).exists():
            raise FileNotFoundError(f"{p} is missing db{ext}.")
    broken = [f.name for f in p.iterdir() if f.is_symlink()]
    if broken:
        raise RuntimeError(f"{p} contains links ({broken[:3]}...); it is not a standalone database. Rebuild it with notebook A.")
    n = n_seqs(db)
    if n != info["sequences"]:
        raise RuntimeError(f"{p}: the index lists {n:,} sequences but db_info.json says {info['sequences']:,}; the copy is incomplete.")
    print(f"{label}: {info['name']} ({info['kind']}), {n:,} sequences, {info['distinct_taxids']:,} distinct taxids, "
          f"made {info['created']} with {info['notebook']}")
    print(f"  source: {info['source']} | taxids: {info['taxids'] or 'n/a'} | self-test: {info['checks']['self_test']}")
    return db, info

DB, INFO = load_database(DATABASE, "DATABASE")
NSEQ = INFO["sequences"]
if NSEQ > 5_000_000 and SEARCH_PRESET != "quick":
    print(f"NOTE: {NSEQ:,} sequences is large for the '{SEARCH_PRESET}' preset on a standard runtime; consider the 'quick' preset first.")

## 4. Search
**What happens.** MMseqs2 compares your query with every sequence of the database. It first finds candidate sequences that share short matching words with your query (the prefilter, controlled by sensitivity), then aligns the candidates and keeps those below the E-value. With more than one iteration it builds a profile from the first round's hits and searches again, which finds more distant relatives. The output has taxonomy columns and the full target sequence. It is saved as a checkpoint, so this is the step you do not want to repeat.

**Expect.** Not measured on real data. Cost grows with the number of sequences searched and the number of iterations, and shrinks with lower sensitivity. Progress lines appear about every 20 seconds, plus a few lines when it passes each internal stage. The output is a hit count and file size. If the search stops with a message about different parameters, you changed something that would force a new (slow) search; the message says how to proceed.

**Parameters that matter.**
- `SEARCH_PRESET`: `quick` (sensitivity 5.7, one iteration, up to 2000 hits) is fast but may miss very remote homologs. `standard` (7.0, one iteration, 10000 hits) and `sensitive` (7.5, two iterations, 20000 hits) find more and take longer. `manual` uses the three `MANUAL_*` values.
- `EVALUE`: the cut-off for reporting a hit. A higher value (for example 1e-3) lets in more distant, less certain hits.
- `SPLIT_MEMORY_LIMIT`: how much RAM one pass may use. The database is read in pieces if it does not fit.
- The query and the database (by its identity in `db_info.json`) are part of the checkpoint key, so changing either needs a new search.

In [ ]:
#@title 4. Search { display-mode: "form" }
PRESETS = {"quick": (5.7, 1, 2000), "standard": (7.0, 1, 10000), "sensitive": (7.5, 2, 20000)}
SENS, ITERS, MAXS = PRESETS.get(SEARCH_PRESET, (MANUAL_SENSITIVITY, MANUAL_ITERATIONS, MANUAL_MAX_SEQS))
FMT = "query,target,fident,qcov,tcov,alnlen,evalue,bits,tlen,taxid,taxname,taxlineage,theader,tseq"
HITS = WORK / "hits.tsv"
HITS_GZ = WORK / "hits.tsv.gz"

sparams = {"query": seq, "database": INFO["id"], "database_name": INFO["name"], "database_sequences": NSEQ, "evalue": EVALUE, "sensitivity": SENS,
           "iterations": ITERS, "max_seqs": MAXS, "format": FMT}
meta = ck_load("search", sparams, expensive=True)
if meta:
    with gzip.open(CK / "search" / "hits.tsv.gz", "rb") as fi, open(HITS, "wb") as fo:
        shutil.copyfileobj(fi, fo)
else:
    print(f"Searching {NSEQ:,} sequences with preset '{SEARCH_PRESET}': sensitivity {SENS}, {ITERS} iteration(s), up to {MAXS} hits.")
    shutil.rmtree(TMP / "search", ignore_errors=True)
    split = f"--split-memory-limit {SPLIT_MEMORY_LIMIT}" if SPLIT_MEMORY_LIMIT else ""
    t0 = time.time()
    sh(f'mmseqs easy-search {QFASTA} {DB} {HITS} {TMP}/search -s {SENS} --num-iterations {ITERS} '
       f'-e {EVALUE} --max-seqs {MAXS} {split} --threads {THREADS} -v 3 --format-output "{FMT}"')
    with open(HITS, "rb") as fi, gzip.open(HITS_GZ, "wb") as fo:
        shutil.copyfileobj(fi, fo)
    ck_save("search", sparams, files={"hits.tsv.gz": HITS_GZ}, extra={"seconds": round(time.time() - t0)})
print("Hit file size (MB):", round(HITS.stat().st_size / 1e6, 2))

## 5. Filter, sample and annotate
**What happens.** The hit table is cleaned: one row per target sequence, only hits with taxonomy, an optional taxon name that must appear in the lineage, query coverage at least `MIN_QCOV`, and identity at least `MIN_FIDENT`. Each filter is logged with the count it leaves. Then `SUBSET_N` limits everything downstream by random sampling with a fixed `SEED`. The result is checkpointed. Taxonomic groups, paralog classes and motif flags are recomputed on every run.

**Expect.** Seconds. Output: the filter log, identity range of the hits and a table of hits per taxonomic group. If almost everything is removed, the log shows which filter did it.

**Parameters that matter.**
- `MIN_QCOV`: the share of your query that must be covered by the alignment. Lower it for a short domain query or fragments, raise it to keep only full-length relatives.
- `SUBSET_N` and `SUBSET_BY`: a quick-test limiter. With `species`, whole species are drawn until at least `SUBSET_N` sequences are reached, so the result can be larger. This keeps each species' sequences together, which the clustering check needs. With `sequence`, exactly `SUBSET_N` are drawn at random, and that breaks up species. Set 0 for no limit.
- `GROUP_RANK` and `EXTRA_CLADES` (Advanced cell): what counts as a "group" in tables and the tree.

In [ ]:
#@title 5. Filter, sample and annotate { display-mode: "form" }
cols = FMT.split(",")
NUMERIC = ["fident", "qcov", "tcov", "alnlen", "evalue", "bits", "tlen"]

def numify(df):
    for c in NUMERIC:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["taxid"] = pd.to_numeric(df["taxid"], errors="coerce").fillna(0).astype(int)
    return df

def species_of(lineage, taxid, taxname):
    m = re.search(r"(?:^|;)s_([^;]*)", lineage)
    return m.group(1) if m else f"{taxname} [{taxid}]"

def subsample(df, n, by, seed):
    if n <= 0 or len(df) <= n:
        return df
    if by == "sequence":
        return df.sample(n, random_state=seed)
    sizes = df.groupby("species").size()
    order = list(np.random.default_rng(seed).permutation(sorted(sizes.index)))
    keep, tot = [], 0
    for sp in order:
        keep.append(sp); tot += int(sizes[sp])
        if tot >= n:
            break
    return df[df.species.isin(keep)]

fparams = {"upstream": KEYS["search"], "taxon_scope": TAXON_SCOPE, "min_qcov": MIN_QCOV, "min_fident": MIN_FIDENT,
           "subset_n": SUBSET_N, "subset_by": SUBSET_BY, "seed": SEED}
meta = ck_load("filter", fparams)
if meta:
    hits = numify(ck_table("filter", "hits"))
    log = [tuple(x) for x in meta["extra"]["log"]]
else:
    hits = pd.read_csv(HITS, sep="\t", names=cols, header=None, quoting=3, dtype=str, keep_default_na=False)
    if hits.empty:
        raise RuntimeError("The search returned no hits. Try a higher-sensitivity preset, a higher E-value, more taxonomic groups or a larger database.")
    hits = numify(hits)
    log = [("alignments in search output", len(hits))]
    hits = hits.sort_values("bits", ascending=False).drop_duplicates("target")
    log.append(("unique targets", len(hits)))
    has_tax = (hits.taxid > 0) & (hits.taxlineage.str.len() > 0)
    if has_tax.sum() == 0:
        raise RuntimeError("No hit carries a taxonomy. The database probably has no taxonomy mapping. "
                           "Use a database built with `mmseqs databases` (UniRef, UniProtKB) or `mmseqs createtaxdb`.")
    hits = hits[has_tax]
    log.append(("with taxonomy", len(hits)))
    if TAXON_SCOPE:
        hits = hits[hits.taxlineage.str.contains(re.escape(TAXON_SCOPE), case=False)]
        log.append((f"in scope '{TAXON_SCOPE}'", len(hits)))
    hits = hits[hits.qcov >= MIN_QCOV]
    log.append((f"query coverage >= {MIN_QCOV}", len(hits)))
    if MIN_FIDENT > 0:
        hits = hits[hits.fident >= MIN_FIDENT]
        log.append((f"identity >= {MIN_FIDENT}", len(hits)))
    if len(hits) < 3:
        raise RuntimeError(f"Only {len(hits)} sequences pass the filters; a tree needs at least 3. "
                           "Relax MIN_QCOV, TAXON_SCOPE or MIN_FIDENT, or use a more sensitive preset.")
    hits["species"] = [species_of(l, t, n) for l, t, n in zip(hits.taxlineage, hits.taxid, hits.taxname)]
    log.append(("species represented", hits.species.nunique()))
    if SUBSET_N > 0:
        hits = subsample(hits, SUBSET_N, SUBSET_BY, SEED)
        log.append((f"after random subset (N={SUBSET_N}, by {SUBSET_BY}, seed {SEED})", len(hits)))
    hits = hits.sort_values("bits", ascending=False).reset_index(drop=True)
    hits["sid"] = [f"s{i:05d}" for i in range(len(hits))]
    ck_save("filter", fparams, tables={"hits": hits}, extra={"log": log})

# Lineage strings look like "-_cellular organisms;d_Eukaryota;-_Opisthokonta;k_Fungi;..." (rank code, underscore, name)
RANK_CODES = {"d": "superkingdom", "k": "kingdom", "p": "phylum", "c": "class", "o": "order",
              "f": "family", "g": "genus", "s": "species", "-": "clade"}
RANK_CHAIN = ["genus", "family", "order", "class", "phylum", "kingdom", "superkingdom"]

def parse_lineage(s):
    out = []
    for part in s.split(";"):
        m = re.match(r"^(.)_(.*)$", part)
        if m:
            out.append((RANK_CODES.get(m.group(1), "clade"), m.group(2)))
    return out

def group_of(lin):
    names = [n for _, n in lin]
    for clade in EXTRA_CLADES:
        if clade in names:
            return clade
    by_rank = {}
    for r, n in lin:
        by_rank.setdefault(r, n)
    for r in RANK_CHAIN[RANK_CHAIN.index(GROUP_RANK):]:
        if r in by_rank:
            return by_rank[r]
    # no ranked ancestor at or above GROUP_RANK: fall back to the first informative lineage entry
    for n in names:
        if n not in ("root", "cellular organisms"):
            return n
    return "Unclassified"

BIG4 = [("Fungi", "Fungi"), ("Metazoa", "Metazoa"), ("Viridiplantae", "Plants"), ("Bacteria", "Bacteria")]
def big4_of(lin):
    names = {n for _, n in lin}
    for key, label in BIG4:
        if key in names:
            return label
    return "Other"

hits["lineage"] = hits.taxlineage.map(parse_lineage)
hits["group"] = hits.lineage.map(group_of)
hits["big4"] = hits.lineage.map(big4_of)

def classify(s):
    for name, rx in PARALOG_RULES.items():
        if re.search(rx, s):
            return name
    return "unclassified"

USE_CLASSES = bool(PARALOG_RULES)
hits["klass"] = hits.tseq.map(classify) if USE_CLASSES else ""
MOTIF_COLS = []
for name, rx in MOTIFS.items():
    col = f"motif:{name}"
    hits[col] = hits.tseq.str.contains(rx, regex=True)
    MOTIF_COLS.append(col)

display(pd.DataFrame(log, columns=["filter", "count"]))
print("Identity to query: min %.2f, median %.2f, max %.2f" % (hits.fident.min(), hits.fident.median(), hits.fident.max()))
display(hits.group.value_counts().rename(f"sequences per {GROUP_RANK}-level group").to_frame())
if USE_CLASSES:
    display(hits.klass.value_counts().rename("hits per class").to_frame())

## 6. Cheap clustering and a label-free sanity check
**What happens.** All filtered sequences (plus the query and any anchors) are clustered with MMseqs2 at several identity thresholds. Two sequences join a cluster if they are similar enough over at least `CLUSTER_COV` of the sequences. For a family with several paralogous subfamilies, distinct sequences of one species should land in different clusters, while each cluster should span many species with about one sequence per species. For each threshold the notebook reports (a) the share of same-species sequence pairs that fall in one cluster, and (b) the share of sequences in clusters with at least `MIN_SPECIES_PER_CLUSTER` species and at most 1.25 sequences per species ("single-copy-like"). With `CLUSTER_PICK = "auto"` the threshold with the highest (b) is used. This is a heuristic that needs no labels. Anchors (`UniProtAccession:label`, fetched from UniProt) only label clusters; they do not define them.

**Expect.** Seconds to minutes for a few thousand sequences (not measured on real data); the clustering reruns once per threshold. Output: one line per threshold, a table of the two measures, three plots (the measures against threshold, the largest clusters by taxonomic group, and a species-by-cluster heatmap) and a cluster summary table.

**How to read it.**
- A high share of same-species pairs in one cluster means the threshold is merging paralogous subfamilies. A very low share of single-copy-like clusters means it is fragmenting orthologs, or the sample has too few sequences per species.
- With UniRef50, two entries of one species cannot be above 50% identical, so thresholds above 0.5 will mostly show singletons.

**Parameters that matter.** `CLUSTER_IDS` (lower thresholds merge more distant sequences), `CLUSTER_COV` (higher keeps domain architecture more uniform), `CLUSTER_PICK`, `MIN_SPECIES_PER_CLUSTER`, `ANCHORS`, and `TREE_CLUSTER_SCOPE` (which clusters go into the tree).

In [ ]:
#@title 6. Clustering { display-mode: "form" }
def fetch_anchor(acc):
    import urllib.request
    with urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=30) as r:
        txt = r.read().decode()
    return "".join(l.strip() for l in txt.splitlines() if not l.startswith(">"))

IDS = sorted({float(x) for x in re.findall(r"[0-9.]+", CLUSTER_IDS)}, reverse=True)
if not IDS:
    raise ValueError("CLUSTER_IDS is empty; give identity thresholds such as 0.6,0.5,0.4,0.3,0.25")
cparams = {"upstream": KEYS["filter"], "ids": IDS, "cov": CLUSTER_COV, "anchors": ANCHORS, "query": seq}
meta = ck_load("cluster", cparams)

def special_row(sid, name, s, group, label):
    return {"sid": sid, "target": name, "taxname": "(query)" if group == "Query" else "(anchor)", "taxid": 0, "group": group,
            "big4": "", "fident": np.nan, "qcov": np.nan, "bits": np.nan, "tseq": s, "tlen": len(s), "label": label,
            "species": ""}

if meta:
    special = ck_table("cluster", "special")
    assign = ck_table("cluster", "assign")
else:
    rows = [special_row("q00000", QNAME, seq, "Query", "")]
    for i, item in enumerate([a.strip() for a in ANCHORS.split(",") if a.strip()], 1):
        acc, _, label = item.partition(":")
        acc, label = acc.strip(), (label.strip() or acc.strip())
        try:
            rows.append(special_row(f"a{i:05d}", f"{acc}", fetch_anchor(acc), "Anchor", label))
            print(f"anchor {label}: {acc}, {len(rows[-1]['tseq'])} residues")
        except Exception as e:
            print(f"WARNING: could not fetch anchor {acc} ({type(e).__name__}: {e}); continuing without it.")
    special = pd.DataFrame(rows)
    allseq = pd.concat([special[["sid", "tseq"]], hits[["sid", "tseq"]]], ignore_index=True)
    CL_FA = WORK / "cluster_input.fa"
    with open(CL_FA, "w") as f:
        for sid, s in zip(allseq.sid, allseq.tseq):
            f.write(f">{sid}\n{s}\n")
    assign = pd.DataFrame({"sid": allseq.sid})
    for t in IDS:
        shutil.rmtree(TMP / "clc", ignore_errors=True)
        pre = WORK / f"cl_{t}"
        sh(f"mmseqs easy-cluster {CL_FA} {pre} {TMP}/clc --min-seq-id {t} -c {CLUSTER_COV} --cov-mode 0 -s 6 --threads {THREADS} -v 1", quiet=True)
        m = dict(l.rstrip("\n").split("\t")[:2][::-1] for l in open(f"{pre}_cluster.tsv"))     # member -> representative
        assign[f"id_{t}"] = [m[s] for s in assign.sid]
        print(f"identity {t}: {assign[f'id_{t}'].nunique()} clusters from {len(assign)} sequences")
    ck_save("cluster", cparams, tables={"special": special, "assign": assign})

special["lineage"] = [[] for _ in range(len(special))]
special["klass"] = [classify(s) if USE_CLASSES else "" for s in special.tseq]
special["tlen"] = pd.to_numeric(special.tlen)
for col in MOTIF_COLS:
    special[col] = [bool(re.search(MOTIFS[col[6:]], s)) for s in special.tseq]
for c in ["fident", "qcov", "bits"]:
    special[c] = pd.to_numeric(special[c], errors="coerce")
special["taxid"] = 0

# --- metrics per threshold
hs = hits.set_index("sid")
def metrics_for(lab, sp):
    df = pd.DataFrame({"cl": list(lab), "sp": list(sp)})
    g = df.groupby(["cl", "sp"]).size()
    gs = df.groupby("sp").size()
    pairs_tot, pairs_same = float((gs * (gs - 1) / 2).sum()), float((g * (g - 1) / 2).sum())
    per = df.groupby("cl").agg(n=("sp", "size"), species=("sp", "nunique"))
    per["spp"] = per.n / per.species
    multi = per.species >= MIN_SPECIES_PER_CLUSTER
    good = multi & (per.spp <= 1.25)
    return {"clusters": len(per), "clusters_multi": int(multi.sum()),
            "merge_pct": 100 * pairs_same / pairs_tot if pairs_tot else np.nan,
            "good_pct": 100 * per.n[good].sum() / len(df),
            "median_seq_per_species": float(per.spp[multi].median()) if multi.any() else np.nan}

arows = assign.set_index("sid")
mrows = []
for t in IDS:
    lab = arows.loc[hits.sid, f"id_{t}"]
    mrows.append({"identity": t, **metrics_for(lab, hits.species)})
met = pd.DataFrame(mrows)
if str(CLUSTER_PICK).strip().lower() == "auto":
    if met.good_pct.max() > 0:
        PICK = float(met.loc[met.good_pct.idxmax(), "identity"])
        how = "highest share of sequences in single-copy-like clusters"
    else:
        PICK = float(met.loc[met.clusters_multi.idxmax(), "identity"])
        how = "no threshold gave single-copy-like clusters; using the one with most clusters spanning several species"
else:
    PICK = min(IDS, key=lambda x: abs(x - float(CLUSTER_PICK)))
    how = "set by CLUSTER_PICK"
print(f"Threshold used downstream: identity {PICK} ({how}).")
display(met.rename(columns={"clusters": "clusters", "clusters_multi": f"clusters with >={MIN_SPECIES_PER_CLUSTER} species",
                            "merge_pct": "same-species pairs in one cluster (%)", "good_pct": "sequences in single-copy-like clusters (%)",
                            "median_seq_per_species": "median sequences per species in those clusters"}).round(2))

# --- name clusters at the chosen threshold, largest first (C1, C2, ...)
col = f"id_{PICK}"
sizes = arows[col].value_counts()
cname = {rep: f"C{i + 1}" for i, rep in enumerate(sizes.index)}
hits["cluster"] = hits.sid.map(lambda s: cname[arows.loc[s, col]])
special["cluster"] = special.sid.map(lambda s: cname[arows.loc[s, col]])
QCLUSTER = special.loc[special.sid == "q00000", "cluster"].iloc[0]
print(f"The query is in cluster {QCLUSTER}"
      + (" together with " + ", ".join(f"{r.label} ({r.cluster})" for r in special[special.group == 'Anchor'].itertuples() if r.cluster == QCLUSTER)
         if (special[special.group == 'Anchor'].cluster == QCLUSTER).any() else "") + ".")

In [ ]:
#@title 6b. Clustering plots and cluster summary { display-mode: "form" }
# Plots for the clustering step
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTHER_C, UNCL_C, NONE_C = "#898781", "#c3c2b7", "#e1e0d9"
INK, INK2 = "#0b0b0b", "#52514e"

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.plot(met.identity, met.merge_pct, "-o", color=SERIES[1], label="same-species pairs sharing a cluster (%)")
ax.plot(met.identity, met.good_pct, "-o", color=SERIES[0], label="sequences in single-copy-like clusters (%)")
ax.axvline(PICK, color=INK2, ls="--", lw=0.8); ax.text(PICK, 128, "used", ha="center", fontsize=8, color=INK2)
ax.set_ylim(-3, 138); ax.set_xlabel("clustering identity threshold"); ax.set_ylabel("%")
ax.legend(fontsize=7, loc="upper right", frameon=False)
ax.set_title("Same-species sequences should separate;\nclusters should be wide and single-copy", fontsize=9, loc="left")
ax = axes[1]
ax.plot(met.identity, met.clusters, "-o", color=SERIES[2], label="clusters")
ax.plot(met.identity, met.clusters_multi, "-o", color=SERIES[6], label=f"clusters with >= {MIN_SPECIES_PER_CLUSTER} species")
ax.axvline(PICK, color=INK2, ls="--", lw=0.8)
ax.set_xlabel("clustering identity threshold"); ax.set_ylabel("number of clusters"); ax.legend(fontsize=7, frameon=False)
ax.set_title("Cluster counts", fontsize=9, loc="left")
for a in axes:
    for s in ("top", "right"): a.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

TOPN = 15
order = [c for c in hits.cluster.value_counts().index][:TOPN]
big_cols = {"Fungi": SERIES[0], "Metazoa": SERIES[1], "Plants": SERIES[2], "Bacteria": SERIES[3], "Other": OTHER_C}
comp = pd.crosstab(hits.cluster, hits.big4).reindex(order).fillna(0)
sp_n = hits.groupby("cluster").species.nunique().reindex(order)
tags = []
for c in order:
    t = c
    s = special[special.cluster == c]
    if (s.group == "Query").any(): t += "  ★ query"
    for lab in s[s.group == "Anchor"].label: t += f"  ◆ {lab}"
    tags.append(t)
fig, ax = plt.subplots(figsize=(9, 0.38 * len(order) + 1.3))
left = np.zeros(len(order))
for g in [g for g in big_cols if g in comp.columns]:
    ax.barh(range(len(order)), comp[g].values, left=left, color=big_cols[g], label=g, height=0.7)
    left += comp[g].values
for i, c in enumerate(order):
    ax.text(left[i] + 0.01 * left.max(), i, f"{int(left[i])} sequence{'s' if left[i] != 1 else ''}, {sp_n[c]} species", va="center", fontsize=7, color=INK2)
ax.set_yticks(range(len(order))); ax.set_yticklabels(tags, fontsize=8); ax.invert_yaxis()
ax.set_xlim(0, left.max() * 1.3); ax.legend(fontsize=7, frameon=False, loc="lower right")
ax.set_title(f"Largest clusters at identity {PICK}: size and taxonomic composition", fontsize=9, loc="left")
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

sp_top = hits.species.value_counts().index[:25]
mat = pd.crosstab(hits.species, hits.cluster).reindex(index=sp_top, columns=order).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(0.55 * len(order) + 3.5, 0.28 * len(sp_top) + 1.5))
im = ax.imshow(mat.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, fontsize=7, rotation=90)
ax.set_yticks(range(len(sp_top))); ax.set_yticklabels([s[:32] for s in sp_top], fontsize=7)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if mat.values[i, j]:
            ax.text(j, i, mat.values[i, j], ha="center", va="center", fontsize=6,
                    color="white" if mat.values[i, j] > mat.values.max() * 0.55 else INK)
ax.set_title("Sequences per species (rows, most sequences first) and cluster (columns).\n"
             "A species should spread over several clusters rather than pile into one.", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

# Cluster descriptions
rows = []
for c in order:
    h = hits[hits.cluster == c]; s = special[special.cluster == c]
    top = h.big4.value_counts(normalize=True)
    r = {"cluster": c, "sequences": len(h), "species": h.species.nunique(), "seq/species": round(len(h) / max(h.species.nunique(), 1), 2),
         "main group": f"{top.index[0]} {top.iloc[0]:.0%}", "median length": int(h.tlen.median()),
         "median identity to query": round(h.fident.median(), 2),
         "contains": ", ".join(["query"] * (s.group == "Query").sum() + list(s[s.group == "Anchor"].label))}
    for mc in MOTIF_COLS:
        r[mc] = f"{h[mc].mean():.0%}"
    rows.append(r)
print("Cluster summary (largest first):")
display(pd.DataFrame(rows))
g = met.loc[met.identity == PICK].iloc[0]
print(f"At identity {PICK}: {g.merge_pct:.0f}% of same-species pairs share a cluster; {g.good_pct:.0f}% of sequences are in single-copy-like clusters.")
if g.merge_pct > 50:
    print("Most same-species pairs share a cluster, so this threshold probably merges paralogous subfamilies. A higher threshold would separate them.")
elif g.good_pct < 20:
    print("Few clusters are wide and single-copy, so this threshold may be fragmenting orthologs, or the sample has too few sequences per species. A lower threshold or a larger SUBSET_N may help.")

## 7. Reduce redundancy
**What happens.** The sequences in scope (all clusters, or only the query's cluster) are clustered once more, this time to remove near-duplicates. The coverage mode lets a fragment join a longer relative. If more than `MAX_TREE_SEQS` representatives remain, the identity threshold is lowered stepwise until the tree is small enough. The query and anchors are always kept.

**Expect.** Seconds to a minute. Output: one line per threshold tried and the final size of the tree set.

**Parameters that matter.**
- `DEDUP_ID`: the starting identity. Lower values remove more near-identical sequences and give a smaller, more even tree.
- `MAX_TREE_SEQS`: the size cap. Alignment and tree time grow quickly with sequence count, and plots become unreadable above about a thousand tips.
- `TREE_CLUSTER_SCOPE`: restrict the tree to the query's cluster to focus on its closest relatives.

In [ ]:
#@title 7. Reduce redundancy { display-mode: "form" }
def write_fasta(rows, path):
    with open(path, "w") as f:
        for sid, s in rows:
            f.write(f">{sid}\n{s}\n")

scope_hits = hits if TREE_CLUSTER_SCOPE == "all clusters" else hits[hits.cluster == QCLUSTER]
if len(scope_hits) < 2:
    raise RuntimeError(f"Only {len(scope_hits)} sequences in scope for the tree. Use TREE_CLUSTER_SCOPE = 'all clusters' or another CLUSTER_PICK.")
rparams = {"upstream": KEYS["cluster"], "cluster_used": PICK, "scope": TREE_CLUSTER_SCOPE, "dedup_id": DEDUP_ID, "max_tree_seqs": MAX_TREE_SEQS}
meta = ck_load("reduce", rparams)
if meta:
    reps = list(ck_table("reduce", "reps").sid)
    USED_ID = float(meta["extra"]["used_id"])
else:
    HITS_FA = WORK / "hits.fa"
    write_fasta(zip(scope_hits.sid, scope_hits.tseq), HITS_FA)

    def cluster_reps(ident):
        prefix = WORK / f"clu_{ident}"
        shutil.rmtree(TMP / "clu", ignore_errors=True)
        sh(f"mmseqs easy-cluster {HITS_FA} {prefix} {TMP}/clu --min-seq-id {ident} -c 0.8 --cov-mode 1 --threads {THREADS} -v 1", quiet=True)
        return [l[1:].split()[0] for l in open(f"{prefix}_rep_seq.fasta") if l.startswith(">")]

    n_special = len(special)
    thresholds = sorted({t for t in [DEDUP_ID, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3] if t <= DEDUP_ID}, reverse=True)
    for t in thresholds:
        reps = cluster_reps(t)
        print(f"identity {t:.2f}: {len(scope_hits)} -> {len(reps)} representatives")
        if len(reps) + n_special <= MAX_TREE_SEQS:
            break
    else:
        best = scope_hits.set_index("sid").loc[reps].sort_values("bits", ascending=False)
        reps = list(best.index[:MAX_TREE_SEQS - n_special])
        print(f"Still above MAX_TREE_SEQS at the lowest threshold; keeping the {len(reps)} best-scoring representatives.")
    USED_ID = t
    ck_save("reduce", rparams, tables={"reps": pd.DataFrame({"sid": reps})}, extra={"used_id": USED_ID})

sel = scope_hits[scope_hits.sid.isin(reps)].copy()
tips = pd.concat([special, sel], ignore_index=True).set_index("sid", drop=False)
tips["label"] = tips["label"].fillna("")
print(f"Tree set: {len(tips)} sequences (query + {int((special.group == 'Anchor').sum())} anchors + {len(sel)} representatives at identity {USED_ID:.2f})")

## 8. Alignment (MAFFT)
**What happens.** MAFFT aligns the tree set with its automatic strategy. Columns that are more than 90% gaps are removed for tree building. The untrimmed alignment is kept.

**Expect.** Minutes for hundreds of sequences (not measured). Output: the number of sequences and columns after trimming, and an occupancy plot (dark = residue, light = gap). A plot with a dark block and mostly light elsewhere means the sequences share only a domain; the trimming then keeps that domain.

**Parameters that matter.** None directly. The sequences come from step 7, so `MAX_TREE_SEQS` and `DEDUP_ID` set the cost. A set mixing full-length proteins and short fragments gives a gappy alignment.

In [ ]:
#@title 8. Alignment { display-mode: "form" }
from Bio import AlignIO, Phylo

SEL_FA, ALN, ALN_TRIM = WORK / "selected.fa", WORK / "aligned.fa", WORK / "aligned_trimmed.fa"
write_fasta(zip(tips.sid, tips.tseq), SEL_FA)
aparams = {"upstream": KEYS["reduce"], "sequences": keyof(sorted(tips.sid), list(tips.tseq))}
meta = ck_load("align", aparams)
if meta:
    shutil.copy(CK / "align" / "aligned.fa", ALN); shutil.copy(CK / "align" / "aligned_trimmed.fa", ALN_TRIM)
else:
    sh(f"mafft --auto --thread {THREADS} --reorder {SEL_FA} > {ALN}", quiet=True)
    aln = AlignIO.read(ALN, "fasta")
    arr = np.array([list(str(r.seq)) for r in aln])
    keep = (arr == "-").mean(axis=0) <= 0.9
    with open(ALN_TRIM, "w") as f:
        for rec, row in zip(aln, arr):
            f.write(f">{rec.id}\n{''.join(row[keep])}\n")
    ck_save("align", aparams, files={"aligned.fa": ALN, "aligned_trimmed.fa": ALN_TRIM})
aln = AlignIO.read(ALN_TRIM, "fasta")
print(f"Alignment: {len(aln)} sequences x {aln.get_alignment_length()} columns after trimming")
full = AlignIO.read(ALN, "fasta")
occ = np.array([[c != "-" for c in str(r.seq)] for r in full])
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.imshow(occ, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_xlabel("alignment column"); ax.set_ylabel("sequence")
ax.set_title("Alignment occupancy (dark = residue, light = gap), untrimmed")
plt.tight_layout(); plt.show()

## 9. Tree
**What happens.** A tree is inferred from the trimmed alignment and rooted. Support values are scaled to 0 to 1. Rooting uses `OUTGROUP_CLASS` (a paralog class, an anchor label or a cluster name) if given, and otherwise the midpoint.

**Expect.** FastTree about a minute for hundreds of sequences. IQ-TREE (model selection plus 1000 ultrafast bootstraps) takes much longer (not measured). Neighbour joining needs no program but is only meant for small sets. Output: the rooting used and the number of tips.

**Parameters that matter.**
- `TREE_METHOD`: `fasttree` for speed, `iqtree` for better support values on small sets, `nj` as a no-install fallback with no support values.
- `OUTGROUP_CLASS`: rooting on a known paralog (for example a 2-oxoacid dehydrogenase E2 of another complex) gives a more meaningful root than the midpoint. The notebook says whether the outgroup tips are monophyletic.

In [ ]:
#@title 9. Tree { display-mode: "form" }
TREE_NWK = WORK / "tree.nwk"
tparams = {"upstream": KEYS["align"], "method": TREE_METHOD}
meta = ck_load("tree", tparams)
if meta:
    shutil.copy(CK / "tree" / "tree.nwk", TREE_NWK)
else:
    if TREE_METHOD == "fasttree":
        sh(f"{find_fasttree()} -lg -gamma {ALN_TRIM} > {TREE_NWK}", quiet=True)
    elif TREE_METHOD == "iqtree":
        exe = find_iqtree()
        thr = "-T AUTO" if exe.endswith("iqtree2") else "-nt AUTO"
        sh(f"{exe} -s {ALN_TRIM} -m MFP -bb 1000 {thr} -pre {WORK}/iq -redo -quiet", quiet=True)
        shutil.copy(WORK / "iq.treefile", TREE_NWK)
    else:
        if len(aln) > 400:
            raise RuntimeError("TREE_METHOD 'nj' is only meant for small sets; use fasttree or lower MAX_TREE_SEQS.")
        from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor
        a = np.array([list(str(r.seq)) for r in aln]); n = len(a)
        D = np.zeros((n, n))
        for i in range(n):
            m = (a != "-") & (a[i] != "-")
            same = ((a == a[i]) & m).sum(axis=1); tot = np.maximum(m.sum(axis=1), 1)
            D[i] = 1 - same / tot
        ids = [r.id for r in aln]
        dm = DistanceMatrix(ids, [list(D[i, :i + 1]) for i in range(n)])
        Phylo.write(DistanceTreeConstructor().nj(dm), TREE_NWK, "newick")
    ck_save("tree", tparams, files={"tree.nwk": TREE_NWK})

sys.setrecursionlimit(20000)
tree = Phylo.read(TREE_NWK, "newick")
for cl in tree.find_clades():
    if cl.confidence is None and cl.name and not cl.is_terminal():
        try: cl.confidence = float(cl.name)
        except ValueError: pass
conf = [c.confidence for c in tree.find_clades() if c.confidence is not None]
if conf and max(conf) > 1.5:                      # IQ-TREE reports 0-100
    for c in tree.find_clades():
        if c.confidence is not None: c.confidence = c.confidence / 100.0

# Rooting: outgroup (class, anchor label or cluster name) if given, else midpoint
rooting = "midpoint"
if OUTGROUP_CLASS:
    og = [t for t in tree.get_terminals()
          if OUTGROUP_CLASS in (tips.loc[t.name, "klass"], tips.loc[t.name, "label"], tips.loc[t.name, "cluster"])]
    if len(og) == 0:
        print(f"No sequences of class, anchor or cluster '{OUTGROUP_CLASS}' in the tree; using midpoint rooting.")
    else:
        mono = tree.is_monophyletic(og)
        tree.root_with_outgroup(og)
        rooting = f"outgroup '{OUTGROUP_CLASS}' ({len(og)} tips, {'monophyletic' if mono else 'NOT monophyletic: root placed at their common ancestor'})"
if rooting == "midpoint":
    tree.root_at_midpoint()
tree.ladderize()
Phylo.write(tree, WORK / "tree_rooted.nwk", "newick")
print("Rooting:", rooting, "| tips:", tree.count_terminals())

## 10. Plot
**What happens.** Draws the tree with rings for taxonomic group, cluster, paralog class (if rules are set) and each motif. The query is a star, anchors are labelled diamonds, and black dots mark nodes with support of at least 0.9 that cover enough tips. The figure is saved as SVG and PNG.

**Expect.** Seconds. The legend gives group and cluster counts. Black tiles in the group ring are the query and anchors, which have no taxonomy.

**Parameters that matter.** `LAYOUT` (circular or rectangular), `GROUP_RANK` (how coarse the group colours are), `MAX_GROUPS` (colours before the rest are grouped as "Other"; more than eight colours become hard to tell apart).

In [ ]:
#@title 10. Plot { display-mode: "form" }
from matplotlib.collections import LineCollection, PatchCollection
from matplotlib.patches import Wedge, Rectangle, Patch
from matplotlib.lines import Line2D


SPECIAL = ("Query", "Anchor")
real = tips[~tips.group.isin(SPECIAL)]

terms = tree.get_terminals()
N = len(terms)

# colours by group size
gcount = real.group.value_counts()
named = [g for g in gcount.index if g != "Unclassified"][:min(MAX_GROUPS, len(SERIES))]
gcolor = {g: SERIES[i] for i, g in enumerate(named)}
if "Unclassified" in gcount.index:
    gcolor["Unclassified"] = UNCL_C
has_other = any(g not in gcolor for g in gcount.index)
if has_other:
    gcolor["Other"] = OTHER_C
def gshow(g):
    return g if g in SPECIAL else (g if g in gcolor else "Other")
tips["gshow"] = tips.group.map(gshow)
real = tips[~tips.group.isin(SPECIAL)]

# clusters: the six largest in the tree get a colour each
ccount = real.cluster.value_counts()
ctop = list(ccount.index[:6])
ccolor = {c: SERIES[(i + 2) % len(SERIES)] for i, c in enumerate(ctop)}
def cshow(c):
    return c if c in ccolor else "other clusters"

classes = [c for c in tips.klass.unique() if c]
kcolor = {}
rev = list(reversed(SERIES))
for i, c in enumerate([c for c in classes if c != "unclassified"]):
    kcolor[c] = rev[i % len(rev)]
if "unclassified" in classes:
    kcolor["unclassified"] = UNCL_C

# tree geometry: depth from root, position index along the tips
sys.setrecursionlimit(20000)
depth, pos = {}, {}
def set_depth(c, d):
    depth[c] = d
    for ch in c.clades:
        set_depth(ch, d + (ch.branch_length or 0.0))
set_depth(tree.root, 0.0)
for i, t in enumerate(terms):
    pos[t] = i
def set_pos(c):
    if c.is_terminal():
        return pos[c]
    ps = [set_pos(ch) for ch in c.clades]
    pos[c] = (min(ps) + max(ps)) / 2.0
    return pos[c]
set_pos(tree.root)
maxd = max(depth[t] for t in terms) or 1.0

# rings: (title, colour per tip, legend entries)
rings = [("Taxonomic group", [gcolor.get(tips.loc[t.name, "gshow"], INK) for t in terms]),
         ("Cluster", [ccolor.get(tips.loc[t.name, "cluster"], UNCL_C) for t in terms])]
if USE_CLASSES:
    rings.append(("Class", [kcolor.get(tips.loc[t.name, "klass"], NONE_C) for t in terms]))
for col in MOTIF_COLS:
    rings.append((f"Motif {col[6:]}", [INK2 if bool(tips.loc[t.name, col]) else NONE_C for t in terms]))

n_major = max(5, int(0.02 * N))
major = [c for c in tree.find_clades(terminal=False)
         if c.confidence is not None and c.confidence >= 0.9 and c.count_terminals() >= n_major and c is not tree.root]
tipgroup = [tips.loc[t.name, "gshow"] for t in terms]
runs, start = [], 0
for i in range(1, N + 1):
    if i == N or tipgroup[i] != tipgroup[start]:
        runs.append((start, i - 1, tipgroup[start])); start = i
runs = [r for r in runs if (r[1] - r[0] + 1) >= max(2, 0.015 * N) and r[2] not in SPECIAL]
qi = next(i for i, t in enumerate(terms) if t.name == "q00000")
anchor_tips = [(i, tips.loc[t.name, "label"]) for i, t in enumerate(terms) if tips.loc[t.name, "group"] == "Anchor"]

def legend_handles():
    hs = [Line2D([], [], linestyle="none", label="Taxonomic group (ring 1)")]
    for g in list(gcolor):
        hs.append(Patch(facecolor=gcolor[g], label=f"{g} ({int((tips.gshow == g).sum())})"))
    hs.append(Patch(facecolor=INK, label="query / anchor (no taxonomy)"))
    hs.append(Line2D([], [], linestyle="none", label=" "))
    hs.append(Line2D([], [], linestyle="none", label="Cluster (ring 2)"))
    for c, col_ in ccolor.items():
        hs.append(Patch(facecolor=col_, label=f"{c} ({int((real.cluster == c).sum())})"))
    hs.append(Patch(facecolor=UNCL_C, label=f"other clusters ({int((~real.cluster.isin(ctop)).sum())})"))
    if USE_CLASSES:
        hs.append(Line2D([], [], linestyle="none", label=" "))
        hs.append(Line2D([], [], linestyle="none", label="Class (ring 3)"))
        for k, c in kcolor.items():
            hs.append(Patch(facecolor=c, label=f"{k} ({int((tips.klass == k).sum())})"))
    if MOTIF_COLS:
        hs.append(Line2D([], [], linestyle="none", label=" "))
        for j, col in enumerate(MOTIF_COLS):
            hs.append(Patch(facecolor=INK2, label=f"{col[6:]} present (ring {len(rings) - len(MOTIF_COLS) + j + 1})"))
            hs.append(Patch(facecolor=NONE_C, label=f"{col[6:]} absent"))
    hs.append(Line2D([], [], linestyle="none", label=" "))
    hs.append(Line2D([], [], marker="*", color=INK, markersize=12, linestyle="none", label=f"query: {QNAME}"))
    if anchor_tips:
        hs.append(Line2D([], [], marker="D", color=INK, markersize=6, linestyle="none", label="anchor (labelled)"))
    hs.append(Line2D([], [], marker="o", color=INK, markersize=4, linestyle="none", label="support >= 0.9 (major nodes)"))
    return hs

def draw_circular():
    th = lambda p: 2 * np.pi * p / N
    xy = lambda r, a: (r * np.cos(a), r * np.sin(a))
    fig, ax = plt.subplots(figsize=(12, 9))
    segs = []
    for c in tree.find_clades(terminal=False):
        ps = [pos[ch] for ch in c.clades]
        a = np.linspace(th(min(ps)), th(max(ps)), max(2, int(abs(th(max(ps)) - th(min(ps))) * 20) + 2))
        x, y = xy(depth[c], a)
        segs.append(np.column_stack([x, y]))
        for ch in c.clades:
            segs.append(np.array([xy(depth[c], th(pos[ch])), xy(depth[ch], th(pos[ch]))]))
    ax.add_collection(LineCollection(segs, colors=INK2, linewidths=0.5, zorder=2))
    R0, w, gap = maxd * 1.04, maxd * 0.05, maxd * 0.012
    ax.add_collection(LineCollection([np.array([xy(depth[t], th(pos[t])), xy(R0, th(pos[t]))]) for t in terms],
                                     colors="#c3c2b7", linewidths=0.3, linestyles=":", zorder=1))
    hw = 180.0 / N
    r_in = R0
    for title, colors in rings:
        patches = [Wedge((0, 0), r_in + w, 360.0 * i / N - hw, 360.0 * i / N + hw, width=w) for i in range(N)]
        ax.add_collection(PatchCollection(patches, facecolors=colors, edgecolors=colors, linewidths=0.3, zorder=3))
        r_in += w + gap
    R_out = r_in - gap
    for s, e, g in runs:
        a = th((s + e) / 2.0); deg = np.degrees(a) % 360
        x, y = xy(R_out + maxd * 0.03, a)
        flip = 90 < deg < 270
        ax.text(x, y, g if len(g) <= 22 else g[:21] + "…", rotation=deg + 180 if flip else deg, rotation_mode="anchor",
                ha="right" if flip else "left", va="center", fontsize=8, color=INK)
    if major:
        mx, my = zip(*[xy(depth[c], th(pos[c])) for c in major])
        ax.scatter(mx, my, s=7, color=INK, zorder=4, linewidths=0)
    qx, qy = xy(R_out + maxd * 0.012, th(qi))
    ax.scatter([qx], [qy], marker="*", s=170, color=INK, edgecolor="white", linewidths=0.8, zorder=6)
    for i, lab in anchor_tips:
        ax_, ay_ = xy(R_out + maxd * 0.012, th(i))
        ax.scatter([ax_], [ay_], marker="D", s=36, color=INK, edgecolor="white", linewidths=0.8, zorder=6)
        tx, ty = xy(R_out + maxd * 0.05, th(i)); deg = np.degrees(th(i)) % 360; flip = 90 < deg < 270
        ax.text(tx, ty, lab, rotation=deg + 180 if flip else deg, rotation_mode="anchor", ha="right" if flip else "left",
                va="center", fontsize=8, color=INK, fontweight="bold")
    lim = R_out + maxd * 0.40
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.axis("off")
    return fig, ax

def draw_rectangular():
    fig, ax = plt.subplots(figsize=(12, max(6, min(40, N * 0.045))))
    segs = []
    for c in tree.find_clades(terminal=False):
        ps = [pos[ch] for ch in c.clades]
        segs.append(np.array([[depth[c], min(ps)], [depth[c], max(ps)]]))
        for ch in c.clades:
            segs.append(np.array([[depth[c], pos[ch]], [depth[ch], pos[ch]]]))
    ax.add_collection(LineCollection(segs, colors=INK2, linewidths=0.6, zorder=2))
    R0, w, gap = maxd * 1.04, maxd * 0.05, maxd * 0.012
    ax.add_collection(LineCollection([np.array([[depth[t], pos[t]], [R0, pos[t]]]) for t in terms],
                                     colors="#c3c2b7", linewidths=0.3, linestyles=":", zorder=1))
    x0 = R0
    for title, colors in rings:
        patches = [Rectangle((x0, i - 0.5), w, 1.0) for i in range(N)]
        ax.add_collection(PatchCollection(patches, facecolors=colors, edgecolors=colors, linewidths=0.3, zorder=3))
        x0 += w + gap
    X_out = x0 - gap
    for s, e, g in runs:
        ax.text(X_out + maxd * 0.02, (s + e) / 2.0, g if len(g) <= 22 else g[:21] + "…", ha="left", va="center", fontsize=8, color=INK)
    if major:
        ax.scatter([depth[c] for c in major], [pos[c] for c in major], s=7, color=INK, zorder=4, linewidths=0)
    ax.scatter([X_out + maxd * 0.012], [qi], marker="*", s=170, color=INK, edgecolor="white", linewidths=0.8, zorder=6)
    for i, lab in anchor_tips:
        ax.scatter([X_out + maxd * 0.012], [i], marker="D", s=36, color=INK, edgecolor="white", linewidths=0.8, zorder=6)
        ax.text(X_out + maxd * 0.05, i, lab, ha="left", va="center", fontsize=8, color=INK, fontweight="bold")
    ax.set_xlim(-maxd * 0.02, X_out + maxd * 0.40); ax.set_ylim(N - 0.5, -0.5); ax.axis("off")
    return fig, ax

fig, ax = (draw_circular if LAYOUT == "circular" else draw_rectangular)()
ax.set_title(f"{QNAME}: {N} sequences, {rooting.split(' (')[0]} rooting, groups at rank {GROUP_RANK}", fontsize=11, color=INK, loc="left")
fig.legend(handles=legend_handles(), loc="center left", bbox_to_anchor=(0.98, 0.5), frameon=False, fontsize=8, handlelength=1.2)
fig.savefig(WORK / "tree.svg", bbox_inches="tight")
fig.savefig(WORK / "tree.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Summary and sanity check
**What happens.** Lists the ten tips closest to the query in the tree and checks that the best-matching database hit is among them. Tabulates sequences per group and writes `annotation.tsv` (one row per tree tip, with taxonomy, cluster, class, motifs) and a short summary text.

**Expect.** Seconds. If the sanity check fails, the alignment, the paralog rule or the tree method is suspect; treat the tree with caution until you know why.

In [ ]:
#@title 11. Summary and sanity check { display-mode: "form" }
qclade = terms[qi]
nn = sorted(((tree.distance(qclade, t), t.name) for t in terms if t is not qclade))[:10]
nn_df = pd.DataFrame([{"tip": s, "tree distance": round(d, 4), "identity to query": tips.loc[s, "fident"],
                       "organism": tips.loc[s, "taxname"], "group": tips.loc[s, "group"], "cluster": tips.loc[s, "cluster"],
                       "anchor": tips.loc[s, "label"],
                       **({"class": tips.loc[s, "klass"]} if USE_CLASSES else {})} for d, s in nn])
print("Closest tips to the query in the tree:")
display(nn_df)
best = real.fident.idxmax()
if best in [s for _, s in nn]:
    print(f"Sanity check passed: the highest-identity hit ({best}) is among the 10 nearest tips in the tree.")
else:
    print(f"WARNING: the highest-identity hit ({best}, identity {tips.loc[best, 'fident']:.2f}) is not among the 10 nearest tips. "
          "Check the alignment, the paralog rule and the tree method before interpreting the tree.")

summary = pd.DataFrame({"hits after filters": hits.group.map(gshow).value_counts(),
                        "sequences in tree": real.gshow.value_counts()}).fillna(0).astype(int)
display(summary)
if USE_CLASSES:
    display(pd.crosstab(real.gshow, real.klass))

top = ", ".join(f"{g} ({n})" for g, n in gcount.head(4).items())
text = (f"{len(hits)} database sequences passed the filters (query coverage >= {MIN_QCOV}"
        + (f", random subset, N={SUBSET_N} by {SUBSET_BY}" if SUBSET_N > 0 else "") + f"). {len(real)} representatives at "
        f"<= {USED_ID:.0%} identity were aligned with MAFFT and used for a {TREE_METHOD} tree ({rooting}). "
        f"The tree contains {gcount.size} groups at rank {GROUP_RANK}; the largest are {top}. "
        f"Clusters at identity {PICK}: {hits.cluster.nunique()}; the query is in {QCLUSTER}. "
        f"The query's nearest tips are {', '.join(nn_df['organism'].head(3))}. "
        "Caveats: UniRef entries take the taxonomy of their representative member, database sampling is uneven across taxa, "
        "and a tree from a database search is a hypothesis about relationships, not a species phylogeny.")
print()
print(textwrap.fill(text, 110))

cols_out = ["sid", "target", "taxid", "taxname", "species", "group", "cluster", "label", "klass", "fident", "qcov", "tlen"] + MOTIF_COLS
ann = tips[cols_out].copy()
hs_ = hits.set_index("sid")
ann["lineage"] = [hs_.loc[t, "taxlineage"] if t in hs_.index else "" for t in ann.sid]
ann.to_csv(WORK / "annotation.tsv", sep="\t", index=False)
(WORK / "summary.txt").write_text(text + "\n")
print("Files in", WORK, ":", ", ".join(sorted(p.name for p in WORK.iterdir() if p.is_file())))

## 12. Stage 2 (optional): classify sequences of a second database
Stage 1 found homologs in one database. Stage 2 asks a different question of a second one, normally a database of *complete* proteomes made in notebook A: which sequence of each species belongs to which stage 1 class? Because every protein of a species is present, a class that is missing from a species is informative, and the number of sequences per species can be compared with what you expect (for example four).

**12a. Load the database.** The same loader as in step 3, applied to `STAGE2_DATABASE`.

**12b. Search and assignment.** Every tip of the stage 1 tree set is a query, labelled with its cluster at the chosen identity (the clusters with the most tips, and every cluster containing the query or an anchor, are kept as classes; the rest become "other", up to `STAGE2_CLASSES`). Each database sequence gets the class of its highest-scoring query. It also gets a margin: the score of its best class minus the score of the best *other* class. A margin below 10% of the best score is flagged as ambiguous, because the sequence resembles two classes about equally. This is a nearest-relative assignment, not a phylogenetic classification, and it can be wrong for sequences far from every query.

**Expect.** Not measured. The search grows with database size and number of queries (up to `MAX_TREE_SEQS`). Output: a table of sequences per species and class, a heatmap of the species with the most hits, a histogram of hits per species, and `stage2_assignment.tsv` and `stage2_species_by_class.tsv` in the work folder.

**Parameters that matter.** `STAGE2_DATABASE` (reference proteomes are one curated set per species; all of UniProtKB for fungi is larger and redundant), `STAGE2_CLASSES` (more classes give finer assignments and a more crowded heatmap), and the stage 1 settings `SEARCH_PRESET`, `EVALUE` and `MIN_QCOV`, which are reused.

In [ ]:
#@title 12a. Stage 2 database { display-mode: "form" }
S2DB, S2INFO = None, None
if not STAGE2:
    print("STAGE2 is not ticked: skipping steps 12a and 12b.")
else:
    S2DB, S2INFO = load_database(STAGE2_DATABASE, "STAGE2_DATABASE")
    S2N = S2INFO["sequences"]

In [ ]:
#@title 12b. Stage 2 search and class assignment { display-mode: "form" }
if STAGE2:
    FMT2 = "query,target,fident,qcov,tcov,bits,evalue,tlen,taxid,taxname,taxlineage"
    ctag = {}
    for r in special.itertuples():
        ctag.setdefault(r.cluster, []).append("query" if r.group == "Query" else r.label)
    def cname2(c):
        return c + (f" ({', '.join(ctag[c])})" if c in ctag else "")
    real_tips = tips[~tips.group.isin(("Query", "Anchor"))]
    keep = set(real_tips.cluster.value_counts().index[:STAGE2_CLASSES]) | set(ctag)
    qclass = tips.cluster.map(lambda c: cname2(c) if c in keep else "other")

    s2params = {"upstream": KEYS["reduce"], "db": S2INFO["id"], "db_name": S2INFO["name"], "db_sequences": S2N, "classes": STAGE2_CLASSES, "evalue": EVALUE,
                "sensitivity": SENS, "iterations": ITERS, "max_seqs": MAXS, "min_qcov": MIN_QCOV}
    meta = ck_load("stage2", s2params, expensive=True)
    if meta:
        assign = ck_table("stage2", "assign")
        for c in ["top_bits", "second_bits", "margin", "fident", "tlen"]:
            assign[c] = pd.to_numeric(assign[c])
        assign["ambiguous"] = assign["ambiguous"] == "True"
    else:
        S2Q, S2H = WORK / "stage2_queries.fa", WORK / "stage2_hits.tsv"
        write_fasta(zip(tips.sid, tips.tseq), S2Q)
        shutil.rmtree(TMP / "search2", ignore_errors=True)
        split = f"--split-memory-limit {SPLIT_MEMORY_LIMIT}" if SPLIT_MEMORY_LIMIT else ""
        sh(f'mmseqs easy-search {S2Q} {S2DB} {S2H} {TMP}/search2 -s {SENS} --num-iterations {ITERS} -e {EVALUE} '
           f'--max-seqs {MAXS} {split} --threads {THREADS} -v 3 --format-output "{FMT2}"')
        h2 = pd.read_csv(S2H, sep="\t", names=FMT2.split(","), header=None, quoting=3, dtype=str, keep_default_na=False)
        if h2.empty:
            raise RuntimeError("Stage 2 search returned no hits.")
        for c in ["fident", "qcov", "tcov", "bits", "evalue", "tlen"]:
            h2[c] = pd.to_numeric(h2[c], errors="coerce")
        h2 = h2[h2.qcov >= MIN_QCOV]
        h2["qclass"] = h2["query"].map(qclass)
        best = h2.sort_values("bits", ascending=False).drop_duplicates("target")
        mat = h2.groupby(["target", "qclass"]).bits.max().astype(float).unstack().fillna(0.0)
        srt = np.sort(mat.values, axis=1)
        top = srt[:, -1]; second = srt[:, -2] if mat.shape[1] > 1 else np.zeros(len(mat))
        assign = best.set_index("target").loc[mat.index, ["taxid", "taxname", "taxlineage", "fident", "tlen"]].copy()
        assign["species"] = [species_of(l, t, n) for l, t, n in zip(assign.taxlineage, assign.taxid, assign.taxname)]
        assign["class"] = mat.idxmax(axis=1).values
        assign["top_bits"], assign["second_bits"] = top, second
        assign["margin"] = top - second
        assign["ambiguous"] = assign.margin < 0.1 * assign.top_bits
        assign = assign.reset_index().rename(columns={"index": "target"})
        ck_save("stage2", s2params, tables={"assign": assign})
    assign.drop(columns=["taxlineage"], errors="ignore").to_csv(WORK / "stage2_assignment.tsv", sep="\t", index=False)

    n_sp = assign.species.nunique()
    print(f"{len(assign):,} sequences from {n_sp:,} species assigned; {assign.ambiguous.mean():.0%} flagged ambiguous (margin below 10% of the best score).")
    order2 = list(assign["class"].value_counts().index)
    sp_class = pd.crosstab(assign.species, assign["class"])[order2]
    sp_class.to_csv(WORK / "stage2_species_by_class.tsv", sep="\t")
    per_sp = sp_class.sum(axis=1)
    print(f"Sequences per species: median {per_sp.median():.0f}, mean {per_sp.mean():.1f}, range {per_sp.min()}-{per_sp.max()}.")
    display(assign["class"].value_counts().rename("sequences").to_frame())

    top_sp = per_sp.sort_values(ascending=False).index[:30]
    m = sp_class.loc[top_sp]
    fig, ax = plt.subplots(figsize=(0.6 * m.shape[1] + 4, 0.28 * len(m) + 1.6))
    ax.imshow(m.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, fontsize=7, rotation=60, ha="right")
    ax.set_yticks(range(len(m))); ax.set_yticklabels([s[:32] for s in m.index], fontsize=7)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            if m.values[i, j]:
                ax.text(j, i, m.values[i, j], ha="center", va="center", fontsize=6,
                        color="white" if m.values[i, j] > m.values.max() * 0.55 else "#0b0b0b")
    ax.set_title("Stage 2: sequences per species (rows, most first) and class (columns)", fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(6, 3))
    vc = per_sp.value_counts().sort_index()
    ax.bar(vc.index, vc.values, color="#2a78d6")
    ax.set_xlabel("sequences assigned per species"); ax.set_ylabel("number of species")
    ax.set_title("How many homologues does each species have?", fontsize=9, loc="left")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.show()

In [ ]:
#@title Download results { display-mode: "form" }
# Download the results
import zipfile
OUT_ZIP = WORK / f"{QNAME}_results.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["tree_rooted.nwk", "tree.svg", "tree.png", "aligned.fa", "aligned_trimmed.fa", "selected.fa",
                 "annotation.tsv", "summary.txt", "hits.tsv.gz", "stage2_assignment.tsv", "stage2_species_by_class.tsv"]:
        if (WORK / name).exists():
            z.write(WORK / name, name)
    if (CK / "manifest.json").exists():
        z.write(CK / "manifest.json", "checkpoint_manifest.json")
print("Wrote", OUT_ZIP)
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT_ZIP))